# Embeddings

This lab is designed to help you solidify your understanding of embeddings by applying them to tasks like semantic similarity, clustering, and building a semantic search system.

### Tasks:
- Task 1: Semantic Similarity Comparison
- Task 2: Document Clustering
- Task 3: Enhance the Semantic Search System


## Task 1: Semantic Similarity Comparison
### Objective:
Compare semantic similarity between pairs of sentences using cosine similarity and embeddings.

### Steps:
1. Load a pre-trained Sentence Transformer model.
2. Encode the sentence pairs.
3. Compute cosine similarity for each pair.

### Dataset:
- "A dog is playing in the park." vs. "A dog is running in a field."
- "I love pizza." vs. "I enjoy ice cream."
- "What is AI?" vs. "How does a computer learn?"


In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sentence pairs
sentence_pairs = [
    ("A dog is playing in the park.", "A dog is running in a field."),
    ("I love pizza.", "I enjoy ice cream."),
    ("What is AI?", "How does a computer learn?")
]

# Compute similarities: encode both sentences and take the cosine similarity
for s1, s2 in sentence_pairs:
    emb = model.encode([s1, s2])
    sim = cosine_similarity([emb[0]], [emb[1]])[0][0]
    print(f"{sim:.3f}  |  '{s1}'  <->  '{s2}'")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

0.522  |  'A dog is playing in the park.'  <->  'A dog is running in a field.'
0.528  |  'I love pizza.'  <->  'I enjoy ice cream.'
0.319  |  'What is AI?'  <->  'How does a computer learn?'


### Questions:
- Which sentence pairs are the most semantically similar? Why?
- Can you think of cases where cosine similarity might fail to capture true semantic meaning?

**Answers:**

- **Most similar pair:** the scores come out *"I love pizza" / "I enjoy ice cream"* ≈ 0.53 and the dog pair ≈ 0.52 — essentially **tied** as the most similar (pizza/ice cream marginally higher), both being everyday statements about a familiar subject. The *"What is AI?" / "How does a computer learn?"* pair scores **lowest** (≈ 0.32). That is a bit counter-intuitive: a human reads the AI/learning pair as clearly related, but the model rates it least similar because the two sentences share little surface structure.
- **Where cosine similarity can fail:** exactly cases like that AI pair — related to a human, low cosine. Also negation (*"I love this"* vs *"I do not love this"* get near-identical vectors but opposite meaning), sarcasm, and the same words used in different senses. Cosine measures embedding closeness, not human-judged relatedness or sentiment polarity.

## Task 2: Document Clustering
### Objective:
Cluster a set of text documents into similar groups based on their embeddings.

### Steps:
1. Encode the documents using Sentence Transformers.
2. Use KMeans clustering to group the documents.
3. Analyze the clusters for semantic meaning.

In [5]:
from sklearn.cluster import KMeans

# Documents to cluster
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?"
]

# Encode documents -> one embedding vector per document
doc_embeddings = model.encode(documents)
print("Embeddings shape:", doc_embeddings.shape)

Embeddings shape: (6, 384)


In [6]:
# Perform KMeans clustering
# The documents split naturally into factual "what is..." vs practical "how do I..." -> 2 clusters
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans.fit_predict(doc_embeddings)
print("Cluster labels:", labels)

Cluster labels: [0 0 0 1 1 1]


In [7]:
# Print cluster assignments grouped by cluster
from collections import defaultdict

clusters = defaultdict(list)
for doc, label in zip(documents, labels):
    clusters[label].append(doc)

for c in sorted(clusters):
    print(f"\nCluster {c}:")
    for d in clusters[c]:
        print("  -", d)


Cluster 0:
  - What is the capital of France?
  - How do I bake a chocolate cake?
  - What is the distance between Earth and Mars?

Cluster 1:
  - How do I change a flat tire on a car?
  - What is the best way to learn Python?
  - How do I fix a leaky faucet?


### Questions:
- How many clusters make the most sense? Why?
- Examine the documents in each cluster. Are they semantically meaningful? Can you assign a semantic "theme" to each cluster?
- Try this exercise with a larger dataset of your choice

**Answers:**

- **Number of clusters:** with `k=2` the split here is **not cleanly thematic** — cluster 0 = {capital of France, bake a cake, Earth–Mars distance}, cluster 1 = {change a tire, learn Python, fix a faucet}. The question phrasing ("what is" vs "how do I") does **not** drive the grouping, and every one of these 6 documents is a different topic.
- **Are the clusters meaningful?** Honestly, not very — the dataset is tiny and topically all-different, so there is no natural 2-way semantic split and KMeans cuts somewhat arbitrarily. That is itself the lesson: clustering needs enough documents per theme and a sensible `k` (elbow / silhouette). With 6 unrelated items, almost every document is its own theme.
- **Larger dataset:** the `model.encode` → `KMeans` pipeline is unchanged; more data per theme plus a tuned `k` would produce clearly interpretable clusters.

## Task 3: Semantic Search System
### Objective:
Create a semantic search engine:
A user provides a query and you search the dataset for semantically relevant documents to return. Return the top 5 results.

### Dataset:
- Use the following set of documents:
    - "What is the capital of France?"
    - "How do I bake a chocolate cake?"
    - "What is the distance between Earth and Mars?"
    - "How do I change a flat tire on a car?"
    - "What is the best way to learn Python?"
    - "How do I fix a leaky faucet?"
    - "What are the best travel destinations in Europe?"
    - "How do I set up a local server?"
    - "What is quantum computing?"
    - "How do I build a mobile app?"


In [8]:
import numpy as np

# Documents dataset
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?",
    "What are the best travel destinations in Europe?",
    "How do I set up a local server?",
    "What is quantum computing?",
    "How do I build a mobile app?"
]

# Compute document embeddings once, so the search function can reuse them
doc_embeddings = model.encode(documents)
print("Embeddings shape:", doc_embeddings.shape)

Embeddings shape: (10, 384)


In [9]:
# Create the search function
# This function should encode the user query and return the top N documents that most resemble it
def semantic_search(query, documents, doc_embeddings, top_n=5):
    # Encode the query into the same vector space as the documents
    query_embedding = model.encode([query])

    # Cosine similarity between the query and every document
    sims = cosine_similarity(query_embedding, doc_embeddings)[0]

    # Indices of the top_n highest-scoring documents (descending)
    top_idx = np.argsort(sims)[::-1][:top_n]
    results = [(documents[i], float(sims[i])) for i in top_idx]

    print(f"Query: {query}\n")
    for rank, (doc, score) in enumerate(results, 1):
        print(f"{rank}. ({score:.3f}) {doc}")
    return results

In [10]:
# Test the search function
query = "Explain programming languages."
semantic_search(query, documents, doc_embeddings)

Query: Explain programming languages.

1. (0.435) What is quantum computing?
2. (0.319) What is the best way to learn Python?
3. (0.110) How do I build a mobile app?
4. (0.091) How do I set up a local server?
5. (0.091) What are the best travel destinations in Europe?


[('What is quantum computing?', 0.4352477490901947),
 ('What is the best way to learn Python?', 0.3187826871871948),
 ('How do I build a mobile app?', 0.11044077575206757),
 ('How do I set up a local server?', 0.09112653881311417),
 ('What are the best travel destinations in Europe?', 0.09064784646034241)]

### Questions:
- What are the top-ranked results for the given queries?
- How can you improve the ranking explanation for users?
- Try this approach with a larger dataset

**Answers:**

- **Top results:** for *"Explain programming languages."* the ranking is 1. *"What is quantum computing?"* (0.44), 2. *"What is the best way to learn Python?"* (0.32), 3. *"How do I build a mobile app?"* (0.11). The intended best match (Python) only comes **second** — quantum computing wins on raw cosine. Note the scores are low overall (max ≈ 0.44), i.e. no document is a strong match for this query.
- **Improving the ranking explanation:** show the cosine score next to each result (done), convert it to a 0–100% relevance, and apply a minimum-similarity threshold so weak matches (the ≈ 0.09 results) are dropped instead of padding the top-5.
- **Larger dataset:** unchanged approach — embed all documents once, then each query is a single `encode` + cosine-similarity sort.